# Topic models

A topic model reads a collection of documents and groups words into topics without any labels. This notebook covers Latent Dirichlet Allocation (LDA): what it assumes, how it is trained, and how to evaluate whether the topics make sense.

In [ ]:
import warnings

import gensim
import matplotlib.pyplot as plt
import numpy as np
from gensim import corpora, models
from gensim.models import CoherenceModel, LdaMulticore
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Intuition

Imagine reading a news article and tagging it "30 % politics, 70 % economics". LDA finds those proportions automatically across a whole corpus, with no labels. Each topic is a probability distribution over the vocabulary: "football" and "goal" appear often in a sport topic, "gradient" and "loss" appear often in a machine learning topic.

LDA says every document is a mixture of topics, and every topic is a mixture of words. Training estimates both mixtures from observed word co-occurrences.

## 2. Maths

LDA is a generative model. To produce a document of length $N$:

1. Sample a topic proportion vector $\theta \sim \text{Dir}(\alpha)$. This is a point on the simplex: each entry is the share of that topic.
2. For each word position $n$:
   - Sample a topic assignment $z_n \sim \text{Categorical}(\theta)$.
   - Sample a word $w_n \sim \text{Categorical}(\beta_{z_n})$, where $\beta_k$ is the word distribution for topic $k$, itself drawn from $\text{Dir}(\eta)$.

Training inverts this process: given the observed words $w$, estimate $\theta$ (per document) and $\beta$ (per topic). The standard algorithm is variational EM, which optimises a lower bound on $\log p(w)$.

The concentration parameter $\alpha$ controls sparsity. A small $\alpha$ pushes each document toward one dominant topic; a large $\alpha$ spreads weight across all topics.

## 3. Plate notation

```
α ──► θ_d ──► z_dn ──► w_dn
                  ▲
              β_k ◄── η
```

Plates (boxes) indicate repetition. The inner plate repeats over $N$ word positions within each document $d$; the outer plate repeats over $D$ documents. The $K$ topic word distributions $\beta_k$ sit outside the document plate because they are shared.

Variables:
- $\alpha$: document-topic Dirichlet prior (hyperparameter).
- $\theta_d$: topic mixture for document $d$ (latent).
- $z_{dn}$: topic assignment for word position $n$ in document $d$ (latent).
- $\beta_k$: word distribution for topic $k$ (latent).
- $\eta$: word-topic Dirichlet prior (hyperparameter).
- $w_{dn}$: observed word.

## 4. Bag-of-words preprocessing

LDA treats each document as a bag of words: order is ignored, only counts matter. The pipeline is tokenise, remove stopwords, lemmatise, then map to integer IDs.

In [ ]:
RAW_DOCS = [
    # machine learning
    "Gradient descent updates model weights by moving opposite to the gradient of the loss.",
    "Overfitting occurs when a model learns noise in the training data and fails to generalise.",
    "Regularisation adds a penalty term to the loss function to reduce model complexity.",
    "A neural network learns feature representations through multiple layers of transformations.",
    "Cross-validation splits data into folds to give an unbiased estimate of generalisation error.",
    "Decision trees split data on feature thresholds that maximise information gain.",
    "Random forests average predictions from many trees to reduce variance.",
    "Backpropagation computes gradients of the loss with respect to every weight in the network.",
    "Learning rate controls the step size during gradient descent optimisation.",
    "Batch normalisation stabilises training by scaling activations within each mini-batch.",
    # football
    "The striker scored two goals in the second half to win the match.",
    "The goalkeeper made a diving save to keep the score level.",
    "Football clubs recruit players during the summer transfer window.",
    "The referee awarded a penalty after the defender brought down the winger.",
    "Counter-attacking football relies on fast transitions from defence to attack.",
    "The midfielder controlled the tempo of the game with precise passing.",
    "Injuries to key players disrupted the team's league campaign.",
    "Pressing high up the pitch forces the opposition into mistakes in their own half.",
    "Set pieces from corners and free kicks are a reliable source of goals.",
    "The manager rotated the squad to manage fatigue across a congested fixture schedule.",
    # cooking
    "Browning meat in a hot pan before braising develops deep flavour through the Maillard reaction.",
    "Pasta should be cooked in well-salted boiling water until al dente.",
    "Whisking egg whites to stiff peaks traps air and gives soufflés their rise.",
    "Marinating chicken in acid and spices tenderises the meat and adds flavour.",
    "Slow roasting vegetables at low heat concentrates sugars and deepens colour.",
    "A roux made from equal parts butter and flour thickens sauces and gravies.",
    "Fresh herbs added at the end of cooking preserve their volatile aromas.",
    "Deglazing a hot pan with wine lifts the caramelised bits into the sauce.",
    "Baking bread requires precise flour-to-water ratios and adequate fermentation time.",
    "Blanching vegetables briefly in boiling water and then cooling in ice water keeps their colour.",
]

print(f"{len(RAW_DOCS)} documents across 3 themes")

In [ ]:
stemmer = PorterStemmer()
lemmatiser = WordNetLemmatizer()


def preprocess(text: str) -> list[str]:
    tokens = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            tokens.append(stemmer.stem(lemmatiser.lemmatize(token)))
    return tokens


processed_docs = [preprocess(doc) for doc in RAW_DOCS]

print("Raw:", RAW_DOCS[0])
print("Processed:", processed_docs[0])

In [ ]:
dictionary = corpora.Dictionary(processed_docs)
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Vocabulary: {len(dictionary)} unique tokens")
print("\nBoW for doc 0 (first 5 entries):")
for word_id, count in bow_corpus[0][:5]:
    print(f"  '{dictionary[word_id]}': {count}")

## 5. TF-IDF corpus

TF-IDF downweights words that appear in almost every document and therefore carry little discriminating information. Words like "data" appear across many ML documents; TF-IDF reduces their weight so topics are driven by more specific vocabulary.

In [ ]:
tfidf = models.TfidfModel(bow_corpus)
tfidf_corpus = tfidf[bow_corpus]

# compare raw BoW vs TF-IDF weights for the first document
bow_weights = dict(bow_corpus[0])
tfidf_weights = dict(tfidf_corpus[0])

print("Token, BoW count, TF-IDF weight (doc 0):")
for wid in list(bow_weights)[:6]:
    print(f"  {dictionary[wid]:20s}  bow={bow_weights[wid]}  tfidf={tfidf_weights.get(wid, 0.0):.3f}")

## 6. LDA with gensim

We ask LDA for 3 topics because that matches the number of themes in our corpus. In practice the right number is unknown and must be chosen by evaluating coherence (section 7).

In [ ]:
lda = LdaMulticore(
    bow_corpus,
    num_topics=3,
    id2word=dictionary,
    passes=20,
    workers=1,
    random_state=SEED,
)

print("Top words per topic:")
for topic_id in range(3):
    words = lda.show_topic(topic_id, topn=10)
    word_str = ", ".join(f"{w}({p:.2f})" for w, p in words)
    print(f"  Topic {topic_id}: {word_str}")

In [ ]:
example_indices = [0, 10, 20]  # one doc from each theme
theme_labels = ["ML", "football", "cooking"]

print("Topic distribution for 3 example documents:")
for idx, label in zip(example_indices, theme_labels):
    dist = lda.get_document_topics(bow_corpus[idx])
    dist_str = ", ".join(f"topic {t}: {p:.2f}" for t, p in sorted(dist, key=lambda x: -x[1]))
    print(f"  [{label}] {dist_str}")

## 7. Coherence

Coherence measures how often the top words of a topic co-occur in the original documents. The C_v metric correlates well with human judgements of topic quality: a coherent topic has words that are semantically related and appear near each other in text.

Plotting coherence against the number of topics gives a rough elbow: beyond the elbow, adding more topics tends to fragment coherent ones rather than finding genuinely new themes.

In [ ]:
k_values = [2, 3, 4, 5]
coherences = []

for k in k_values:
    model_k = LdaMulticore(
        bow_corpus,
        num_topics=k,
        id2word=dictionary,
        passes=20,
        workers=1,
        random_state=SEED,
    )
    cm = CoherenceModel(
        model=model_k,
        texts=processed_docs,
        dictionary=dictionary,
        coherence="c_v",
    )
    coherences.append(cm.get_coherence())
    print(f"k={k}: coherence={coherences[-1]:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(k_values, coherences, marker="o")
plt.xlabel("Number of topics")
plt.ylabel("Coherence (C_v)")
plt.title("Topic coherence vs number of topics")
plt.xticks(k_values)
plt.tight_layout()
plt.show()

## 8. What breaks

**Short documents.** A tweet or headline gives LDA little evidence to estimate $\theta_d$. The posterior stays close to the prior and topic assignments are noisy.

**Identical priors for every document.** A single $\alpha$ assumes every document has the same tendency toward topic sparsity. Hierarchical LDA relaxes this but adds complexity.

**Number of topics is a hyperparameter.** LDA does not decide how many topics exist. Coherence and perplexity give signal, but the final choice is a judgement call. Models with 3 and 4 topics can both be valid depending on what granularity is useful.

**Topics are not labelled.** The model outputs word distributions. Naming a topic "football" requires reading the top words and deciding. Labels are human annotations, not model outputs.

**Word order is ignored.** LDA treats each document as a bag of counts. Phrases like "not good" and "good" are indistinguishable. Topic models built on n-grams or on sentence embeddings handle this, at the cost of a larger vocabulary or a different model class entirely.